## Statistics Trend analysis

In this section, the following possible trends will be investigated by looking at randomly sampled hierarchies of various heights.

**Trends improving with more levels:**
- Monthly and daily reconciled error mean and variance (error statistics).
- Monthly and daily reconciled error q90 (error statistics).

**Trends worsening with more levels:**
- Monthly and daily reconciled error median (error statistics).

In [1]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS, TWO_DAYS, FOUR_DAYS, PeriodicAggregation, ArbitraryAggregation, Aggregation
from python.temporal_hierarchy import TemporalHierarchy
from python.sampled_hierarchies import SampledHierarchies

/home/eartaam/egyetem/semester_six/forecast_research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
available_levels: list[Aggregation] = [
    TWO_DAYS,
    FOUR_DAYS,
    WEEKDAYS,
    WEEKENDS,
    WEEKS,
    FORTNIGHTS,
    ArbitraryAggregation('8Days1', [1] * 8 + [0] * 20),
    ArbitraryAggregation('8Days2', [0] * 8 + [1] * 8 + [0] * 12),
    ArbitraryAggregation('8Days3', [0] * 16 + [1] * 8 + [0] * 4),
    ArbitraryAggregation('16Days', [1] * 16 + [0] * 12),
    ArbitraryAggregation('12Days', [0] * 16 + [1] * 12),
    PeriodicAggregation('11L', 1, 1, True), # Skip every other
    PeriodicAggregation('11R', 1, 1, False),
    PeriodicAggregation('43L', 4, 3, True), # 4 Days on, 3 skip
    PeriodicAggregation('43R', 4, 3, False),
    PeriodicAggregation('34L', 3, 4, True), # 3 Days on, 4 skip
    PeriodicAggregation('34R', 3, 4, False),
    PeriodicAggregation('61L', 6, 1, True), # 6 Days on, 1 skip
    ArbitraryAggregation('AllWeekends', [0,0,0,0,0,1,1] * 4),
    ArbitraryAggregation('AllWeekdays', [1,1,1,1,1,0,0] * 4),
    ArbitraryAggregation('AllExtendedWeekends', [0,0,0,0,1,1,1] * 4),
    ArbitraryAggregation('AllExtendedWeekdays', [1,1,1,1,1,1,0] * 4),
    ArbitraryAggregation('10', [1, 0] * 14), # Zig zag
    ArbitraryAggregation('01', [0, 1] * 14)
]

In [3]:
em = LoadedErrorMetrics('metrics/sample_hiers_errors.csv')

sh = SampledHierarchies(available_levels, 3, 10)

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy='Minimal'), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=h))
    for h in sh.hierarchies
])

### Error trends

The first kind of trends investigated will those ones which are connected to statistics of the raw NRMSE error values. I will use a predefined method to calculate the error statistics for each of the individual hierarcies, and arrange them in a matrix shape, where rows correspond to groups, and columns correspond to hierarchies of the same height.

During the calculation of the individual statistics, the top 0.1 percent of values are ignored for more meaningful mean and variance values across the statistics. Our suspected statistics trends do not involve extreme outliers anyway (e.g. max), and they were found previously in the same way, with the outliers sorted out.

In [4]:
mon_rec_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=MONTHS, forecast_type=RECONCILED_FORECASTS),
                                       df_filter=lambda df: df[df < np.quantile(df.values, 0.999)])
day_rec_stats_matrix = sh.error_matrix(em, sh.index_matrix(granularity=DAYS, forecast_type=RECONCILED_FORECASTS),
                                       df_filter=lambda df: df[df < np.quantile(df.values, 0.999)])

After fetching the individual statistics, I will aggregate them, per height value, to essentially use the sampled hierarchies' error statistics as samples of a single distribution (from which we now have 10 samples).

In [5]:
mon_aggs = sh.aggregate_over(stats_matrix=mon_rec_stats_matrix,
                  group_by='heights',
                  stats=['mean', 'median', 'q90', 'std'],
                  aggs=['mean', 'median', 'min', 'max'])

day_aggs = sh.aggregate_over(stats_matrix=day_rec_stats_matrix,
                  group_by='heights',
                  stats=['mean', 'median', 'q90', 'std'],
                  aggs=['mean', 'median', 'min', 'max'])

#### Daily and monthly mean

First I will look at the suspected trend which would state that, more hierarchy levels improve the mean of the daily and the monthly errors. I will plot bar charts of various aggregates of the means of individual hierarchies, first for the monthly error mean:

In [6]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'mean'])

As we can see, the mean aggregate of the mean statistic is consistently decreasing by adding more and more hierarchy levels (at least up to the point of the limit of our available levels).

In [7]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'median'])

The same is true for the median aggregate of the different means.

In [8]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'min'])

In [9]:
plots.plot_aggregate_statistics(mon_aggs['mean', 'max'])

It is very nice to see that the trend still holds for even the minimum and maximum of the individual means.

Nextg we look at the same plots but for daily granularity errors.

In [10]:
plots.plot_aggregate_statistics(day_aggs['mean', 'mean'])

As we can see, by looking at the mean aggregate, it seems like the trend still holds even for daily granularuity, however, the range is much smaller.

In [11]:
plots.plot_aggregate_statistics(day_aggs['mean', 'median'])

With the median aggregate, there is a small bump from 3 to 4, but overall the trend still holds, but it is still much weaker than at monthly granularity.

In [12]:
plots.plot_aggregate_statistics(day_aggs['mean', 'max'])

In [13]:
plots.plot_aggregate_statistics(day_aggs['mean', 'min'])

When looking at the maximum or minimum aggregate, the trend does start to fall apart, which hints at the fact that at daily granularity this improvement by having more and more hierarchy levels is not as significant, and also inconsistent.

In conclusion, the trend does seem correct for top level error, and it might be true (but weaker) for bottom level error.

TODO

In [36]:
sh.trend_check_per_group(mon_rec_stats_matrix, 'mean', 'decrease')

array([0.71428571, 0.85714286, 0.71428571, 0.71428571, 0.71428571,
       0.71428571, 0.85714286, 0.85714286, 0.85714286])

In [15]:
plots.plot_sampled_error_matrix(sh, 'mean', stats_matrix=day_rec_stats_matrix)

#### Daily and monthly median

Next we look at the suspected trend stating that with having more hierarchy levels, the median of the errors at daily granularity increases. First we look at monthly granularity.

In [16]:
plots.plot_aggregate_statistics(mon_aggs['median', 'mean'])

In [17]:
plots.plot_aggregate_statistics(mon_aggs['median', 'median'])

Looking at the mean aggregate of the median statistics as monthly granularity, it seems like the trend could be true, but even so it would be a very slight trend (i.e. little range). The median aggregate shows no promise either, but there the small differences are inconsistent as well.

Let us look at daily granularity error median statistics

In [18]:
plots.plot_aggregate_statistics(day_aggs['median', 'mean'])

In [19]:
plots.plot_aggregate_statistics(day_aggs['median', 'median'])

Looking at the mean and median aggregate of the median statistic of daily errors, the trend we suspected does seem true. 

In [20]:
plots.plot_aggregate_statistics(day_aggs['median', 'min'])

In [21]:
plots.plot_aggregate_statistics(day_aggs['median', 'max'])

When looking at the minimum and maximum aggregates, the results are promising, showing a trend of increasing maximum and minimum median error, with slight variation.

Overall, we can say that our trend suspicion was correct for the bottom level errors: having more hierarchy levels increases bottom level error median value. The same was not proven for top level errors.

#### Daily and monthly variance

Next we look at the trends stating that having more levels improves/decreases the variance of monthly and daily errors. First we look at the monthly statistics

In [22]:
plots.plot_aggregate_statistics(mon_aggs['std', 'mean'])


In [23]:
plots.plot_aggregate_statistics(mon_aggs['std', 'median'])

When looking at the mean or median aggregates of the variance statistics, we can see that the trend holds true until about 15 hierarchy levels, then the variance start increasing slightly.

In [24]:
plots.plot_aggregate_statistics(mon_aggs['std', 'min'])


In [25]:
plots.plot_aggregate_statistics(mon_aggs['std', 'max'])

When looking at the minimum/maximum aggregate, the trend starts to fall apart even more.

Next we look at daily granularity.

In [26]:
plots.plot_aggregate_statistics(day_aggs['std', 'mean'])

In [27]:
plots.plot_aggregate_statistics(day_aggs['std', 'median'])

When looking at the aggregates of daily error variances, the trend disappears.

Overall, there is a slight trend where having more hierarchy levels decreases top level error variance, but it seemingly only works up to a point, and the same trend does not appear at bottom level aggregates.

#### Daily and monthly P90

Next we investigate the suspected trend that having more hierarchy levels decreases the p90 value of both daily and monthly level errors. First we look at daily granularity

In [28]:
plots.plot_aggregate_statistics(day_aggs['q90', 'mean'])

In [29]:
plots.plot_aggregate_statistics(day_aggs['q90', 'median'])

When looking at the mean and median aggregate of the p90 values, the trend is apparent, and it also has quite a high magnitude.

In [30]:
plots.plot_aggregate_statistics(day_aggs['q90', 'min'])

In [31]:
plots.plot_aggregate_statistics(day_aggs['q90', 'max'])

The trend is not as consistent with the maximum and minimum values, but it still more or less holds.

Next we look at monthly granularity

In [32]:
plots.plot_aggregate_statistics(mon_aggs['q90', 'mean'])

In [33]:
plots.plot_aggregate_statistics(mon_aggs['q90', 'median'])

The effect of the trend is apparent at monthly granularity as well, at the same magnitude

In [34]:
plots.plot_aggregate_statistics(mon_aggs['q90', 'min'])

In [35]:
plots.plot_aggregate_statistics(mon_aggs['q90', 'max'])

At monthly granularity, the trend still holds by looking at the minimum and maximum of the p90 values.

Overall, the trend stating that by having more hierarchy levels, the p90 values of the top and bottom level errors decrease, holds true in both cases, however it is more consistent and powerful (i.e. higher range of aggregates) at top level granularity.

### Conclusions

**Trends confirmed:**
- Monthly mean decreases with more levels — strong and consistent across all aggregates.
- Daily mean decreases with more levels — weaker than monthly, falls apart at min/max aggregates.
- Daily median increases with more levels — consistent across all aggregates.
- Monthly q90 decreases with more levels — strong and consistent, holds even at min/max aggregates.
- Daily q90 decreases with more levels — apparent at mean/median aggregates, less consistent at min/max.
- Monthly variance decreases with more levels — holds up to about 15 levels, then starts increasing; falls apart at min/max aggregates.

**Trends not confirmed:**
- Monthly median does not show a meaningful trend — very slight at best, inconsistent.
- Daily variance does not show a trend — no pattern visible in the aggregates.